# P5-1 · GĐ 1 · Diffusion tạo ảnh thế nào

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

## Lý thuyết

In [ ]:
import torch, math
T = 1000
betas = torch.linspace(1e-4, 0.02, T)
alphas = 1 - betas
alpha_bars = torch.cumprod(alphas, dim=0)     # tích lũy: ᾱ₁, ᾱ₂, ..., ᾱ₁₀₀₀

def q_sample(x0, t, eps):
    ab = alpha_bars[t].unsqueeze(1)            # (B,) -> (B,1) để nhân với x0 có shape (B,2)
    return ab.sqrt() * x0 + (1 - ab).sqrt() * eps

In [ ]:
import torch.nn as nn

def time_emb(t, dim=32):
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half) / half)
    ang = t.float().unsqueeze(1) * freqs.unsqueeze(0)          # (B, half)
    return torch.cat([ang.sin(), ang.cos()], dim=1)             # (B, dim)

class Denoiser(nn.Module):
    def __init__(self, h=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2 + 32, h), nn.SiLU(), nn.Linear(h, h), nn.SiLU(),
                                 nn.Linear(h, h), nn.SiLU(), nn.Linear(h, 2))
    def forward(self, x, t):
        return self.net(torch.cat([x, time_emb(t)], dim=1))     # ghép xₜ với embedding của t

s = torch.rand(4000).sqrt() * 3 * math.pi                       # 4000 điểm trên xoắn ốc
data = torch.stack([s * s.cos(), s * s.sin()], 1) / 5

model = Denoiser(); opt = torch.optim.Adam(model.parameters(), lr=1e-3)
for step in range(3000):
    x0 = data[torch.randint(0, len(data), (256,))]
    t = torch.randint(0, T, (256,))
    eps = torch.randn_like(x0)
    loss = ((model(q_sample(x0, t, eps), t) - eps) ** 2).mean()
    opt.zero_grad(); loss.backward(); opt.step()

In [ ]:
@torch.no_grad()
def sample(model, n=400):
    x = torch.randn(n, 2)                                    # x₁₀₀₀: nhiễu thuần
    for t in reversed(range(T)):                             # 999, 998, ..., 0
        eps_hat = model(x, torch.full((n,), t))
        x = (x - betas[t] / (1 - alpha_bars[t]).sqrt() * eps_hat) / alphas[t].sqrt()
        if t > 0:
            x = x + betas[t].sqrt() * torch.randn_like(x)    # nhiễu mới, trừ bước cuối
    return x

## Bài tập 4: Lịch nhiễu trên một ảnh thật {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
x0 = load_digits().images[0] / 8 - 1          # 8 × 8, giá trị trong [-1, 1]
T = 1000
betas = np.linspace(1e-4, 0.02, T)
# TODO: alpha_bar; t_nhieu (bước đầu tiên √ᾱ < 0.1); hàm q_sample(x0, t, eps); vẽ 4 mức nhiễu

In [ ]:
alpha_bar = np.cumprod(1 - betas)
t_nhieu = int(np.argmax(np.sqrt(alpha_bar) < 0.1))

def q_sample(x0, t, eps):
    return np.sqrt(alpha_bar[t]) * x0 + np.sqrt(1 - alpha_bar[t]) * eps

rng = np.random.default_rng(0)
eps = rng.standard_normal(x0.shape)
fig, axes = plt.subplots(1, 5, figsize=(9, 2.2))
for ax, t in zip(axes, [0, 50, 200, 500, 999]):
    ax.imshow(q_sample(x0, t, eps) if t else x0, cmap="gray_r"); ax.axis("off")
    ax.set_title(f"t={t}\n√ᾱ={np.sqrt(alpha_bar[t]):.2f}", fontsize=9)
print("tín hiệu dưới 10% từ bước", t_nhieu)

tín hiệu dưới 10% từ bước 673


## Bài tập 5: Diffusion đầu tiên trên điểm 2D {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import numpy as np, torch, torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
torch.manual_seed(0)
X = torch.tensor(make_moons(2000, noise=0.05, random_state=0)[0], dtype=torch.float32)
X = (X - X.mean(0)) / X.std(0)
T = 100
betas = torch.linspace(1e-4, 0.05, T); alphas = 1 - betas; abar = torch.cumprod(alphas, 0)
net = nn.Sequential(nn.Linear(3, 128), nn.SiLU(), nn.Linear(128, 128), nn.SiLU(), nn.Linear(128, 2))
# TODO: train 3000 bước (batch 256, Adam 2e-3); sample 1000 điểm; mean_nn_gen, mean_nn_train

In [ ]:
opt = torch.optim.Adam(net.parameters(), lr=2e-3)
for step in range(3000):
    x0 = X[torch.randint(0, len(X), (256,))]
    t = torch.randint(0, T, (256,))
    eps = torch.randn_like(x0)
    a = abar[t].unsqueeze(1)
    xt = a.sqrt() * x0 + (1 - a).sqrt() * eps
    loss = ((net(torch.cat([xt, t.unsqueeze(1) / T], 1)) - eps) ** 2).mean()
    opt.zero_grad(); loss.backward(); opt.step()

with torch.no_grad():
    x = torch.randn(1000, 2)
    for t in reversed(range(T)):
        e = net(torch.cat([x, torch.full((1000, 1), t / T)], 1))
        x = (x - betas[t] / (1 - abar[t]).sqrt() * e) / alphas[t].sqrt()
        if t > 0:
            x = x + betas[t].sqrt() * torch.randn_like(x)
d = torch.cdist(x, X)
mean_nn_gen = d.min(1).values.mean().item()
so_trung = (d.min(1).values < 1e-4).sum().item()
dd = torch.cdist(X[:500], X[500:])
mean_nn_train = dd.min(1).values.mean().item()
fig, ax = plt.subplots(figsize=(5, 3.6))
ax.scatter(X[:, 0], X[:, 1], s=2, alpha=0.3, label="dữ liệu")
ax.scatter(x[:, 0], x[:, 1], s=2, c="red", alpha=0.5, label="sinh ra"); ax.legend()
print(f"loss {loss.item():.3f} | điểm sinh → dữ liệu gần nhất: {mean_nn_gen:.3f} | trùng khít: {so_trung}")

loss 0.436 | điểm sinh → dữ liệu gần nhất: 0.069 | trùng khít: 0
